# 04. Augmentation: синтетика как «продолжение» реального ряда

**Q2.** Если синтетика плохо заменяет реальные данные (B2 vs B1), может быть она хотя бы **дополнит** их полезным образом?

**Протокол B3:**

1. Берём TimeGAN, обученный на real train (тот же, что в B2).
2. Семплируем синтетический ряд длины `len(train)=2515` («искусственная история»).
3. Склеиваем: `[synth_TimeGAN] + [real train]` → выборка длиной 5030 точек.
4. Обучаем GARCH(1,1)-t на этом расширенном train, фиксируем параметры.
5. Walk-forward на real test (2020–2024).

Почему мы клеим синтетику **до** реального train, а не после: индекс — формальный, GARCH видит только последовательность ε² и σ². Если синтетика «похожа» на real train, она работает как дополнительная «история» того же режима.

In [1]:
import sys, json
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))
ART = ROOT / "artifacts"

params_real = json.loads((ART / "garch_params_real.json").read_text(encoding="utf-8"))
params_aug  = json.loads((ART / "garch_params_aug.json").read_text(encoding="utf-8"))

p_table = pd.DataFrame({"B1 (real)": params_real, "B3 (augmented)": params_aug}).T
p_table["alpha+beta"] = p_table["alpha"] + p_table["beta"]
p_table.round(5)

,mu,omega,alpha,beta,nu,dist,alpha+beta
B1 (real),0.086791,0.025294,0.175878,0.812557,4.873854,t,0.988435
B3 (augmented),0.033632,0.030691,0.141026,0.826108,75.959895,t,0.967134


In [2]:
summary = pd.read_csv(ART / "metrics_summary.csv")
mask = summary["branch"].isin(["B1_real", "B3_aug"])
cols = ["branch","RMSE_abs_r","MAE_abs_r","QLIKE","MZ_a","MZ_b","MZ_R2",
        "VaR5_violations","VaR5_rate","VaR5_kupiec_p",
        "VaR1_violations","VaR1_rate","VaR1_kupiec_p"]
summary.loc[mask, cols].set_index("branch").round(4)

,RMSE_abs_r,MAE_abs_r,QLIKE,MZ_a,MZ_b,MZ_R2,VaR5_violations,VaR5_rate,VaR5_kupiec_p,VaR1_violations,VaR1_rate,VaR1_kupiec_p
branch,,,,,,,,,,,,
B1_real,0.8867,0.6334,1.0898,0.1645,0.9002,0.3042,99,0.0787,0.0000,20,0.0159,0.0528
B3_aug,0.8686,0.6197,1.0896,0.0903,1.0486,0.2990,84,0.0668,0.0093,32,0.0254,0.0000


**Что показала аугментация (B3 vs B1):**

* α+β сохранилось ≈ 0.97 (vs 0.99 у B1) — persistence остаётся сильным, в отличие от B2-TimeGAN, где β был «убит» полностью.
* RMSE(σ̂, |r|) **0.869 < 0.887** — небольшое, но устойчивое улучшение.
* MZ-наклон **b ≈ 1.05** против 0.90 у B1 — прогноз лучше калиброван (теоретическое b=1).
* Доля нарушений VaR(5%): **6.7% против 7.9%** у B1 — модель перестала так сильно недооценивать хвостовой риск. На VaR(1%) аугментация, наоборот, ухудшила: 2.5% vs 1.6% — здесь сигнал смешанный.
* **Резюме Q2:** аугментация TimeGAN-синтетикой, добавленной к реальному train, **не ломает** GARCH (как это делал B2) и даёт **умеренное улучшение** калибровки прогноза волатильности и VaR(5%). Эффект небольшой, но систематический по нескольким метрикам сразу. Это согласуется с литературой: синтетика полезнее как «дополнение», а не как «замена».